# Test version compatibility

Covers all logical branches of `_confirm_version` and its helper functions, across different server configurations.

In [1]:
from firefly_client import FireflyClient

# only needed for the mock scenario
from firefly_client._server_compat import is_server_compatible, FIREFLY_VERSION_KEY
from unittest.mock import patch, MagicMock

In [2]:
# Uncomment for debugging outputs
FireflyClient._debug = True

In [3]:
def pprint_confirm_version(result):
    """For pretty-printing the result of _confirm_version() for debugging."""
    print(f"compatible:     {result['compatible']}")
    print(f"server_version: {result['server_version']!r}")
    try:
        raw = result['response'].json()
    except Exception:
        raw = result['response'].text[:200]
    print(f"raw response:   {raw}")

## Compatible server
Creating a FireflyClient instance should succeed without errors.

In [4]:
fc = FireflyClient.make_client(url='http://localhost:8080/firefly', launch_browser=False)

DEBUG: new instance: http://localhost:8080/firefly


`_confirm_version()` should return `compatible=True` and the server version string from the `Firefly Version` field in the response `data`.

In [5]:
pprint_confirm_version(fc._confirm_version())

compatible:     True
server_version: '2026.1-DEV:FIREFLY-1331-version-validation_f9ee'
raw response:   {'success': True, 'data': {'Application Version': '2026.1-DEV:FIREFLY-1331-version-validation_f9ee', 'Built On': 'Wed Apr 29 13:25:44 PDT 2026', 'Git commit': 'f9ee5f1b6', 'Firefly Version': '2026.1-DEV:FIREFLY-1331-version-validation_f9ee'}}


## Incompatible server version

Intercept the version endpoint response and substitute an old version string, then verify that creating a FireflyClient instance raises `ValueError` with a message explaining how to resolve the issue.

We don't have such a server available, so we mock `_confirm_version` to simulate this scenario.

In [6]:
INCOMPATIBLE_VERSION = '2025.3.2'  # below MIN_SERVER_VERSION (2025.4)

mock_resp_incompat = MagicMock()
mock_resp_incompat.status_code = 200
mock_resp_incompat.json.return_value = {'success': True, 'data': {FIREFLY_VERSION_KEY: INCOMPATIBLE_VERSION}}

ver_incompat = {
    'compatible': is_server_compatible(INCOMPATIBLE_VERSION),
    'server_version': INCOMPATIBLE_VERSION,
    'response': mock_resp_incompat,
}

In [7]:
# Patch _confirm_version at class level so the mock takes effect inside make_client()
with patch.object(FireflyClient, '_confirm_version', return_value=ver_incompat):
    FireflyClient.make_client(url='http://localhost:8080/firefly/', launch_browser=False)

ValueError: Version of the provided Firefly server http://localhost:8080/firefly/ is not compatible with this version of firefly_client.
  Server version: 2025.3.2
  Required: >=2025.4
  Please use the URL of a compatible Firefly server


`_confirm_version()` should return `compatible=False` and the old version string.

In [8]:
pprint_confirm_version(ver_incompat)

compatible:     False
server_version: '2025.3.2'
raw response:   {'success': True, 'data': {'Firefly Version': '2025.3.2'}}


## Version unknown

When the version endpoint returns `success: false`, is unreachable, or the version string is absent from the response, `_confirm_version()` falls back to `compatible=True` to preserve backward compatibility with servers that predate the version endpoint. Creating a FireflyClient instance should succeed.

In [9]:
fc_old = FireflyClient.make_client(url='https://irsa.ipac.caltech.edu/irsaviewer/', launch_browser=False)

DEBUG: new instance: https://irsa.ipac.caltech.edu/irsaviewer/


`_confirm_version()` should return `compatible=True` and `server_version=None`.

In [10]:
pprint_confirm_version(fc_old._confirm_version())

compatible:     True
server_version: None
raw response:   {'success': False, 'error': {}}
